In [0]:
# ============================================================
#  dim_date  —  static date dimension (2020-01-01 → 2030-12-31)
# ============================================================
from pyspark.sql import functions as F

dbutils.widgets.dropdown(name="environment", defaultValue="dev",
                         choices=["dev","qa","prd"], label="select Environment")
env = dbutils.widgets.get("environment")
gold_tbl = f"saleslake_{env}.gold_{env}.dim_date"

df = (spark.sql("SELECT explode(sequence(to_date('2020-01-01'), "
                "to_date('2030-12-31'), interval 1 day)) AS full_date")
      .withColumn("date_sk",       F.date_format("full_date", "yyyyMMdd").cast("int"))
      .withColumn("day_of_month",  F.dayofmonth("full_date"))
      .withColumn("month_number",  F.month("full_date"))
      .withColumn("month_name",    F.date_format("full_date", "MMMM"))
      .withColumn("quarter",       F.quarter("full_date"))
      .withColumn("year",          F.year("full_date"))
      .withColumn("day_of_week",   F.dayofweek("full_date"))
      .withColumn("day_name",      F.date_format("full_date", "EEEE"))
      .withColumn("week_of_year",  F.weekofyear("full_date"))
      .withColumn("is_weekend",    F.col("day_of_week").isin(1,7))
      .withColumn("is_month_end",  F.last_day("full_date") == F.col("full_date"))
      .withColumn("fiscal_year",
           F.when(F.month("full_date") >= 4, F.year("full_date") + 1)
            .otherwise(F.year("full_date")))
      .withColumn("fiscal_quarter",
           F.when(F.month("full_date").between(4,6),  F.lit("1"))
            .when(F.month("full_date").between(7,9),  F.lit("2"))
            .when(F.month("full_date").between(10,12), F.lit("3"))
            .otherwise(F.lit("4")))
      .withColumn("created_ts",    F.current_timestamp())
      .select("date_sk","full_date","day_of_month","day_name","week_of_year",
              "month_number","month_name","quarter","year",
              "is_weekend","is_month_end",
              "fiscal_year","fiscal_quarter","created_ts"))

df.write.format("delta").mode("overwrite").saveAsTable(gold_tbl)
print(f"dim_date populated: {df.count()} rows")
spark.table(gold_tbl).limit(5).display()